# SpamShield AI - Baseline Model

First pass: train and evaluate DistilBERT on SMS Spam Collection.

In [ ]:
# For Colab: uncomment below to mount Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install -q torch transformers datasets scikit-learn pandas matplotlib seaborn wordcloud nltk

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertModel, DistilBertTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Dataset

In [ ]:
# Local path
DATA_PATH = 'data/raw/SMSSpamCollection'

# For Colab: use this instead
# DATA_PATH = '/content/drive/My Drive/data/smsspamcollection/SMSSpamCollection'

df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['label', 'text'], encoding='latin-1')
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
df['label_name'] = df['label'].map({0: 'Ham', 1: 'Spam'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label_name'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

df['text_length'] = df['text'].str.len()
for label, color, name in [(0, '#2ecc71', 'Ham'), (1, '#e74c3c', 'Spam')]:
    subset = df[df['label'] == label]
    axes[1].hist(subset['text_length'], bins=40, alpha=0.6, color=color, label=name)
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Preprocess

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

df['cleaned'] = df['text'].apply(clean_text)
df[['text', 'cleaned', 'label']].head()

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    df['cleaned'].tolist(), df['label'].tolist(),
    test_size=0.15, random_state=42, stratify=df['label']
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
TOKENIZER = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
MAX_LEN = 128

def tokenize(texts):
    return TOKENIZER(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors=None)

train_enc = tokenize(X_train)
val_enc = tokenize(X_val)
test_enc = tokenize(X_test)
print(f'Tokenized {len(train_enc["input_ids"])} training samples')

## 3. Dataset & Model

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class SpamClassifier(nn.Module):
    def __init__(self, num_labels=2, dropout=0.3):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(self.distilbert.config.hidden_size, 256)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        x = self.dropout1(pooled)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout2(x)
        return self.fc2(x)

## 4. Train

In [ ]:
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5

train_loader = DataLoader(SpamDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SpamDataset(val_enc, y_val), batch_size=BATCH_SIZE)

model = SpamClassifier().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_f1': []}
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            logits = model(input_ids, attention_mask)
            val_loss += criterion(logits, labels).item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average='binary')

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')

model.load_state_dict(torch.load('best_model.pt', weights_only=True))
print(f'\nBest Val F1: {best_f1:.4f}')

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'o-', label='Train')
axes[0].plot(epochs, history['val_loss'], 's-', label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['val_accuracy'], 'o-', color='#2ecc71')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')

axes[2].plot(epochs, history['val_f1'], 's-', color='#e74c3c')
axes[2].set_title('Validation F1')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## 6. Evaluate on Test Set

In [ ]:
test_loader = DataLoader(SpamDataset(test_enc, y_test), batch_size=BATCH_SIZE)

model.eval()
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].cpu().numpy())

print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
fp = [(X_test[i], y_prob[i]) for i in range(len(y_test)) if y_true[i] == 0 and y_pred[i] == 1]
fn = [(X_test[i], y_prob[i]) for i in range(len(y_test)) if y_true[i] == 1 and y_pred[i] == 0]

print(f'False Positives: {len(fp)}')
for text, prob in sorted(fp, key=lambda x: x[1], reverse=True)[:5]:
    print(f'  [{prob:.3f}] {text[:80]}')

print(f'\nFalse Negatives: {len(fn)}')
for text, prob in sorted(fn, key=lambda x: x[1])[:5]:
    print(f'  [{prob:.3f}] {text[:80]}')

## 8. Manual Predictions

In [ ]:
def predict(text):
    enc = TOKENIZER(text, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
    input_ids = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    conf = probs[0][pred].item()
    return 'Spam' if pred == 1 else 'Ham', conf

test_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Congratulations! You've won a $1000 gift card. Click here to claim now!",
    "URGENT: Your account has been compromised. Verify your password immediately.",
    "Can you pick up some milk on your way home?",
    "FREE FREE FREE! Win a car by texting WIN to 80085",
    "Meeting moved to 3pm. See you there.",
    "You have been selected for a cash prize! Call now!",
    "Thanks for dinner last night, it was great!"
]

for msg in test_messages:
    label, conf = predict(msg)
    marker = 'SPAM' if label == 'Spam' else 'HAM '
    print(f'[{marker}] ({conf:.2%}) {msg[:70]}')

## 9. Save Model

In [ ]:
SAVE_PATH = 'models/final'
!mkdir -p $SAVE_PATH

torch.save(model.state_dict(), f'{SAVE_PATH}/model.pt')
TOKENIZER.save_pretrained(SAVE_PATH)

with open(f'{SAVE_PATH}/history.json', 'w') as f:
    json.dump(history, f)

print(f'Model saved to {SAVE_PATH}')